In [4]:
from sklearn.model_selection import train_test_split,cross_val_score,KFold,GridSearchCV
from sklearn.linear_model import LinearRegression,LogisticRegression,RidgeClassifier
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [6]:
df=pd.read_csv("diabetes.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [16]:
X=df.drop(["Outcome"],axis=1).values
y=df["Outcome"].values

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)


models={"Ridge Classifier":RidgeClassifier(),
        "Logistic Regression":LogisticRegression(),
        "KNeighborsClassifier":KNeighborsClassifier()}

results={}

for name,model in models.items():

    kf=KFold(n_splits=10,random_state=42,shuffle=True)

    cv_scores=cross_val_score(model,X_train_scaled,y_train,cv=kf,scoring="accuracy")

    results[name]=cv_scores



rdf=pd.DataFrame({
    'Model':results.keys(),
    'Mean Accuracy':[scores.mean() for scores in results.values()],
    'Std Dev':[scores.std() for scores in results.values()]
})

rdf=rdf.sort_values(by="Mean Accuracy",ascending=False)


params={"C":[0.1,1,10],"l1_ratio":[0]}

tunedmodel=GridSearchCV(LogisticRegression(class_weight="balanced"),params,cv=10)
tunedmodel.fit(X_train_scaled,y_train)
y_pred=tunedmodel.predict(X_test_scaled)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

